# InternVL -> SegFormer-B0 Pilot 蒸馏

从 GitHub 固定提交运行：470 张训练、75 张验证、50 张固定测试。先缓存 InternVL Teacher 语义概率和视觉特征，再使用相同设置训练 SegFormer-B0 Baseline 与 KD Student。

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('当前 Kaggle notebook 未分配 GPU')
print('GPU:', torch.cuda.get_device_name(0))
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
repo_dir = Path('/kaggle/working/internvl-pv-distillation')
repo_url = 'https://github.com/Zh0416/internvl-pv-distillation.git'
fixed_commit = '83fc3b39f31140055431f7f25ba05362ac0a88bf'

subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', fixed_commit], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != fixed_commit:
    raise RuntimeError(f'Git commit不一致: expected={fixed_commit}, actual={actual_commit}')
print('Repository commit:', actual_commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(repo_dir / 'requirements.txt')], check=True)
os.chdir(repo_dir)

In [ ]:
config_path = repo_dir / 'configs/kaggle_pv_pilot_distill.yaml'
subprocess.run([sys.executable, 'scripts/08_prepare_pilot_manifest.py', '--config', str(config_path)], check=True)
manifest_path = Path('/kaggle/working/outputs/reports/pilot_manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
all_keys = [row['cache_key'] for rows in manifest['splits'].values() for row in rows]
if len(all_keys) != 595 or len(set(all_keys)) != 595:
    raise RuntimeError(f'Pilot划分数量或唯一性错误: total={len(all_keys)}, unique={len(set(all_keys))}')
print(json.dumps({'counts': manifest['counts'], 'unique_samples': len(set(all_keys)), 'hard_positive_ratio_ceiling': manifest['hard_positive_ratio_ceiling']}, ensure_ascii=False, indent=2))

In [ ]:
subprocess.run([sys.executable, 'scripts/09_extract_pilot_teacher.py', '--config', str(config_path)], check=True)

In [ ]:
subprocess.run([sys.executable, 'scripts/10_train_pilot_student.py', '--config', str(config_path), '--variant', 'baseline'], check=True)

In [ ]:
subprocess.run([sys.executable, 'scripts/10_train_pilot_student.py', '--config', str(config_path), '--variant', 'kd'], check=True)

In [ ]:
subprocess.run([sys.executable, 'scripts/11_compare_pilot.py'], check=True)
reports_dir = Path('/kaggle/working/outputs/reports')
teacher = json.loads((reports_dir / 'pilot_teacher_report.json').read_text(encoding='utf-8'))
comparison = json.loads((reports_dir / 'pilot_comparison.json').read_text(encoding='utf-8'))
summary = {
    'repository_commit': fixed_commit,
    'teacher': {
        'success_count': teacher['success_count'],
        'failure_count': teacher['failure_count'],
        'semantic_disabled_count': teacher['semantic_disabled_count'],
        'metrics': teacher['metrics'],
    },
    'baseline_test': comparison['baseline']['test'],
    'kd_test': comparison['kd']['test'],
    'delta': comparison['delta'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Comparison report:', reports_dir / 'pilot_comparison.json')